
# Solution key for PCA

## Learning goals
By working through this notebook, students will learn to:

- explain PCA conceptually from two perspectives:
  - variance maximization
  - decorrelation / change of coordinates
- derive the PCA eigenvalue problem
- carry out a manual PCA on a small dataset
- implement PCA in Python
- interpret explained variance, scores, and loadings
- use PCA in a classification pipeline
- discuss strengths and limitations of PCA for bioinformatics

## Dataset
We use the **Breast Cancer Wisconsin Diagnostic Dataset**, loaded directly from `scikit-learn`.  
Although this is not a high-throughput omics dataset, it behaves like a realistic biomedical feature table:
- rows = patient samples
- columns = quantitative biological measurements
- target = benign vs malignant tumor class


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

plt.rcParams["figure.figsize"] = (7, 5)



---
## Problem 1. Conceptual foundations of PCA

### 1A. Variance maximization view

**Question 1. Why is maximizing variance useful?**

**Answer:**  
When data are projected onto a direction with larger variance, the projected points are more spread out.  
This often means that the direction preserves more of the structure of the original dataset. If we projected onto a direction with very small variance, many points would collapse together and we would lose information.

**Question 2. What does the first principal component (PC1) represent?**

**Answer:**  
PC1 is the direction in feature space along which the projected data have the largest possible variance, subject to the direction having unit length. It is the single most informative linear direction in the dataset in the sense of captured variance.

**Question 3. Why must PC2 be orthogonal to PC1?**

**Answer:**  
If PC2 were not orthogonal to PC1, it would overlap with information already captured by PC1. By requiring orthogonality, PCA ensures that each new component captures a new, nonredundant direction of variation.

### 1B. Decorrelation view

**Question 4. What does it mean for variables to be uncorrelated?**

**Answer:**  
Two variables are uncorrelated if their covariance is zero. This means they do not show a linear tendency to increase or decrease together.

**Question 5. Why does diagonalizing the covariance matrix achieve decorrelation?**

**Answer:**  
The covariance matrix in the transformed coordinate system becomes diagonal when the new axes are chosen as eigenvectors of the original covariance matrix. A diagonal covariance matrix means all off-diagonal covariances are zero, so the transformed variables are uncorrelated.

**Question 6. Why does PCA produce uncorrelated principal components?**

**Answer:**  
PCA projects the data onto orthogonal eigenvectors of the covariance matrix. In that new coordinate system, the covariance matrix is diagonal, so the component scores are uncorrelated.

### Covariance formula

For centered data matrix $X$ with rows as samples and columns as features, a common covariance form is

$$
\Sigma = \frac{1}{n-1} X^T X
$$

where:
- $X$ = centered data matrix
- $X^T X$ = feature-by-feature covariance-like cross-product
- $n-1$ = normalization factor

Covariance is central to PCA because PCA looks for directions that explain variation, and covariance measures how the features vary and co-vary.



---
## Problem . PCA implementation in Python on a biomedical dataset


In [ ]:

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Shape of feature matrix:", X.shape)
print("Class names:", data.target_names)
X.head()


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca_full.explained_variance_ratio_))],
    "Explained Variance Ratio": pca_full.explained_variance_ratio_,
    "Cumulative Explained Variance": np.cumsum(pca_full.explained_variance_ratio_)
})
explained.head(10)


In [ ]:

plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
         np.cumsum(pca_full.explained_variance_ratio_), marker="o")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance by PCA Components")
plt.grid(True)
plt.show()

n90 = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= 0.90) + 1
print("Number of PCs needed to explain at least 90% variance:", n90)



### Answer

- The dataset has **569 samples** and **30 features**
- PCA should be applied after standardization because the features are on different scales
- The first few PCs usually capture a large fraction of the total variance
- In this dataset, about **90% of the variance** is explained by the first several components, not just the first two

This shows that PCA compresses the dataset, but some information is still distributed across multiple directions.



---
## Problem 5. PCA visualization


In [ ]:

pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "PC1": X_pca_2[:, 0],
    "PC2": X_pca_2[:, 1],
    "Class": y.map({0: "malignant", 1: "benign"})
})

for label in ["malignant", "benign"]:
    subset = plot_df[plot_df["Class"] == label]
    plt.scatter(subset["PC1"], subset["PC2"], label=label, alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection onto First Two Principal Components")
plt.legend()
plt.grid(True)
plt.show()

print("Explained variance ratio of PC1:", pca_2.explained_variance_ratio_[0])
print("Explained variance ratio of PC2:", pca_2.explained_variance_ratio_[1])
print("Combined:", pca_2.explained_variance_ratio_.sum())



### Answer

The scatter plot often shows noticeable separation between benign and malignant tumors in the PC1-PC2 plane.

**Interpretation:**
- PCA itself does **not** use class labels
- Yet if the dominant sources of variation are biologically related to disease status, class separation may appear naturally in PCA space
- Good separation suggests that some major variation in the measured biomarkers is associated with tumor type

However, PCA is not guaranteed to separate classes well because it optimizes variance, not class discrimination.



---
## Problem 4. PCA loadings and biological interpretation


In [ ]:

loadings = pd.DataFrame(
    pca_full.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(X.shape[1])]
)

pc1_top = loadings["PC1"].abs().sort_values(ascending=False).head(10)
pc2_top = loadings["PC2"].abs().sort_values(ascending=False).head(10)

print("Top features contributing to PC1:")
print(pc1_top)
print()
print("Top features contributing to PC2:")
print(pc2_top)



### Answer

A **loading** tells us how strongly an original feature contributes to a principal component.

- A **large positive loading** means the feature strongly contributes in the positive direction of that PC
- A **large negative loading** means the feature strongly contributes in the negative direction of that PC
- A **large absolute loading** means the feature is important for that component

### Biological interpretation
If a feature has a high absolute loading in PC1, then that feature is strongly associated with the dominant pattern of variation across samples.

### Can PCA be used for biomarker discovery?
PCA can be useful as an **exploratory tool** for suggesting important features, but it should not by itself be treated as definitive biomarker discovery because:
- PCA ignores class labels
- high variance does not always mean biological importance
- a biologically meaningful feature might have modest variance



---
## Problem 5. PCA + classification pipeline


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Baseline logistic regression without PCA
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
y_prob_base = baseline.predict_proba(X_test)[:, 1]

baseline_results = {
    "Accuracy": accuracy_score(y_test, y_pred_base),
    "ROC AUC": roc_auc_score(y_test, y_prob_base)
}
baseline_results


In [ ]:

results = []

for k in [2, 5, 10, X.shape[1]]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=k)),
        ("model", LogisticRegression(max_iter=5000))
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "k": k,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df



### Answer

PCA may help classification when:
- it reduces noise
- it removes redundancy
- the signal is concentrated in a few dominant directions

PCA may hurt classification when:
- class-separating information lies in lower-variance directions
- too many informative features are discarded
- the chosen number of components is too small

This is an important lesson: **good compression is not always the same as good prediction**.



---
## Problem 6. PCA vs original feature space



### Answer

**Dimensionality reduction vs information loss:**  
PCA reduces the number of variables, which simplifies modeling and visualization, but some information is inevitably discarded when only a subset of PCs is kept.

**Noise reduction:**  
If low-variance directions mostly contain noise, removing them can improve robustness.

**Overfitting:**  
In high-dimensional settings, PCA can reduce the risk of overfitting by replacing many correlated variables with a smaller set of orthogonal components.

### Why is PCA useful in high-dimensional omics data?

In omics datasets such as transcriptomics, proteomics, or metabolomics:
- the number of features can be very large
- many features are correlated
- visualization in the original space is impossible
- dimension reduction can reveal major biological patterns such as disease subtype, batch effects, or treatment response

So PCA is often one of the first tools used for exploratory analysis in bioinformatics.



---
## Problem 7. Limitations of PCA in bioinformatics

### 1. PCA is linear — why is this a limitation?

PCA can only capture linear combinations of features. If the true biological structure is nonlinear, PCA may miss important patterns.

### 2. PCA ignores class labels — why can this be problematic?

The directions of largest variance are not necessarily the directions that best separate classes. A low-variance direction might still be highly predictive.

### 3. PCA maximizes variance, not biological relevance — explain.

A feature can vary a lot because of technical noise, measurement artifacts, or irrelevant biological variation. High variance alone does not guarantee biological importance.

### 4. Give one example where PCA might fail in bioinformatics.

If disease groups differ in a subtle nonlinear way, or if a strong batch effect dominates the variance, PCA may mainly capture technical variation rather than disease biology.



## Final summary

### Main conceptual takeaways
1. PCA finds directions that maximize variance.
2. PCA also finds a new coordinate system in which the transformed variables are uncorrelated.
3. The principal directions are eigenvectors of the covariance matrix.
4. The amount of variance explained by each PC is given by its eigenvalue.
5. PCA is very useful for exploratory analysis and visualization in bioinformatics, especially in high-dimensional settings.
6. PCA does not use labels, so it should not be confused with a supervised classification method.

### Main practical takeaways
1. Standardization is usually necessary before PCA.
2. The first two PCs are often useful for visualization, but may not capture all important information.
3. Loadings help interpret which original variables contribute to each principal component.
4. PCA can help or hurt prediction depending on whether the discarded directions contain useful class information.
